# 01 - Running HealthSynth

This notebook shows the basic ways to run HealthSynth and inspect the generated datasets.

You will learn:

- how to generate data using the Python API
- how output formats work
- how YAML profiles change behavior
- how to inspect CSV and DuckDB outputs

In [ ]:
from healthsynth.generator import generate

In [ ]:
datasets = generate(
    hcps=250,
    years=2,
    output_dir="../output/notebook_basic_csv",
    seed=42,
    output_format="csv",
)

In [ ]:
datasets.keys()

In [ ]:
for name, df in datasets.items():
    if name != "_timings":
        print(name, df.shape)

In [ ]:
datasets["_timings"]

In [ ]:
datasets["product"]

In [ ]:
datasets["hcp_master"].head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hcp_master = datasets["hcp_master"]

specialty_counts = (
    hcp_master["specialty"]
    .value_counts()
    .reset_index()
)

specialty_counts.columns = ["specialty", "hcp_count"]

plt.figure(figsize=(9, 5))
plt.bar(specialty_counts["specialty"], specialty_counts["hcp_count"])
plt.title("HCP Count by Specialty")
plt.xlabel("Specialty")
plt.ylabel("HCP Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
duckdb_datasets = generate(
    hcps=250,
    years=2,
    output_dir="../output/notebook_duckdb",
    seed=42,
    output_format="duckdb",
)

In [ ]:
import duckdb

con = duckdb.connect("../output/notebook_duckdb/healthsynth.duckdb")
con.execute("SHOW TABLES").fetchdf()

In [ ]:
con.execute("""
SELECT
    p.product_name,
    SUM(rx.nrx) AS total_nrx,
    SUM(rx.trx) AS total_trx
FROM prescriptions rx
JOIN product p
    ON rx.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_nrx DESC
""").fetchdf()

In [ ]:
oncology = generate(
    config_path="../configs/profiles/oncology_training.yaml",
    output_dir="../output/notebook_oncology",
)

In [ ]:
oncology["hcp_master"]["specialty"].value_counts(normalize=True)

In [ ]:
oncology["call_activity"]["channel"].value_counts(normalize=True)

## What you learned

In this notebook, you generated HealthSynth data using:

- direct Python parameters
- CSV output
- DuckDB output
- YAML configuration profiles

This is the foundation for the later notebooks, where we will explore specific commercial analytics use cases.